In [18]:
import os

print("CWD:", os.getcwd())
print("scripts exists:", os.path.exists("./scripts"))
print("csv exists:", os.path.exists("./winequality-red.csv"))


CWD: /home/ec2-user/SageMaker/Problem_2_SageMaker
scripts exists: True
csv exists: False


In [6]:
# Installs only if SageMaker import is broken in this kernel image.
# OKay great no error this time even though it didn't throw the exception for the install either but good news nonetheless
import sys
import subprocess

try:
    import boto3
    import sagemaker
    from sagemaker.sklearn.estimator import SKLearn
except Exception as e:
    print("Initial import failed, upgrading SageMaker SDK:", e)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "sagemaker"])
    import boto3
    import sagemaker
    from sagemaker.sklearn.estimator import SKLearn

print("sagemaker version:", sagemaker.__version__)


sagemaker version: 2.257.0


In [7]:
session = sagemaker.Session()
region = session.boto_region_name
bucket = session.default_bucket()

# Use explicit role for this notebook instance.
role = "arn:aws:iam::047557961235:role/service-role/AmazonSageMaker-ExecutionRole-20260221T114423"

print("region:", region)
print("bucket:", bucket)
print("role:", role)


region: us-east-2
bucket: sagemaker-us-east-2-047557961235
role: arn:aws:iam::047557961235:role/service-role/AmazonSageMaker-ExecutionRole-20260221T114423


In [14]:
# First check if the data/.csv is being found or not?
local_csv = "data/winequality-red.csv"  # or your actual path
if os.path.exists(local_csv):
    print(f"CSV found: {local_csv}")
else:
    raise FileNotFoundError(f"CSV not found: {local_csv}")




CSV found: data/winequality-red.csv


In [12]:
# Break out the next part here now.... this is where we'll actually upload the data into the S3 bucket
s3_prefix = "ana680/wine-quality/data"
train_s3_uri = session.upload_data(path=local_csv, key_prefix=s3_prefix)
print("train_s3_uri:", train_s3_uri)
# Cool it worked, and ALSO note we can go here to actually navigate to S3 within the AWS console:
# https://us-east-2.console.aws.amazon.com/s3/home?region=us-east-2#
# I don't quite know why I have 3 different buckets showing here but I see the one with the wine quality data is this one:
# sagemaker-us-east-2-047557961235
# (Other 2 might be from the failed attempt at sagemaker studio setup, which was blocked by quota since I'm still on waitlist I guess)

train_s3_uri: s3://sagemaker-us-east-2-047557961235/ana680/wine-quality/data/winequality-red.csv


In [28]:
# Part A: built-in SKLearn estimator (no custom container).
script_path = "train_2a.py"


sk_estimator = SKLearn(
    entry_point=script_path,
    source_dir="scripts",
    role=role,
    instance_count=1,
    # instance_type="ml.m4.xlarge", # Sample code used this ml.m4.xlarge but on notebook creation we didn't use this we used ml.t2.medium
    # Nope the ml.t2.medium isn't correct either... upon further investigation that is ONLY for the notebook running environment, AWS uses a different instance for training
    # To figure out which ones I DO have a quote >0 for, I have to go into the AWS Console - Service Quotas - Amazon SageMaker AI and search for training job usage
    # Of course..... they have me set to a 0 quota for absolutely every single possible instance when I search "for training job usage" and so I am hard blocked now
    # I requested the quota increase for ml.m4.xlarge but now it's stuck on PENDING status so who knows how long this will take.... this platform is exceptionally frustrating to use
    # Also tried (I think) the smallest one I see which can be used when I search "for training job usage" which was ml.t3.large and that also went to support case :-/
    #instance_type="ml.t3.large",
    instance_type="ml.m4.xlarge",
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=session,
)

print(sk_estimator)
# print(dir(sk_estimator))

# Oh it worked it finally worked! Had to open a business plus account to get my quota requests prioritized and that seems to have increased the quotas automatically I think
# It took 286 seconds on ml.t3.large so now try with his ml.m4.xlarge to see the timing differences... I'd imagine there's some tradeoff to slower/cheaper takes more time
# Only took 134 seconds on ml.m4.xlarge... I'm sure we could examine the prices per second for these 2 and figureout which approach is cheaper; maybe later... 

In [29]:
sk_estimator.fit({"train": train_s3_uri})
print("Training complete")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-scikit-learn-2026-02-22-18-14-53-105


2026-02-22 18:14:54 Starting - Starting the training job...
2026-02-22 18:15:08 Starting - Preparing the instances for training...
2026-02-22 18:15:29 Downloading - Downloading input data.........
2026-02-22 18:17:25 Training - Training image download completed. Training in progress../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-02-22 18:17:26,383 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-02-22 18:17:26,386 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-02-22 18:17:26,389 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-02-22 18:17:26,403 sagemaker_sklearn_co

In [30]:
print(sk_estimator.latest_training_job.name)
print(sk_estimator.model_data)

sagemaker-scikit-learn-2026-02-22-18-14-53-105
s3://sagemaker-us-east-2-047557961235/sagemaker-scikit-learn-2026-02-22-18-14-53-105/output/model.tar.gz


In [31]:
predictor_2a = sk_estimator.deploy(initial_instance_count=1, instance_type="ml.t2.medium")
print("Endpoint:", predictor_2a.endpoint_name)
# In the output look for the 7 underscores and the ! which means that it's successful
# Oh actually update... his slide has 7 but I'm already on 8 and it's still going so I think it'll just periodically give us underscores until it's done... variable count
# i.e. look for the ! and that's when we know it's done... not necessarily 7 underscores... also can just print from the object at the end to double check too... 
# When examinig the endpoint list using "for endpoint usage" in the service quotas I see they do have smaller ones like for example: ml.t2.medium
# Only user is my (and later professor) so will start with this smaller version as I think the smallest possible with 1 instance is PROBABLY all we need...


INFO:sagemaker:Creating model with name: sagemaker-scikit-learn-2026-02-22-18-39-09-933
INFO:sagemaker:Creating endpoint-config with name sagemaker-scikit-learn-2026-02-22-18-39-09-933
INFO:sagemaker:Creating endpoint with name sagemaker-scikit-learn-2026-02-22-18-39-09-933


--------!Endpoint: sagemaker-scikit-learn-2026-02-22-18-39-09-933


In [39]:
print("Endpoint:", predictor_2a.endpoint_name)
print("Model data:", sk_estimator.model_data)

# The endpoing was created and storing the output name of it just in case for future reference in the comments
# If we for example re-ran... just like the model train it'll overwrite the object in this notebook but the object itself still exists on the "back end" inside AWS
# Endpoint: sagemaker-scikit-learn-2026-02-22-18-39-09-933

#Also note that we can go to the "Endpoints" section inside of "Deployments and Inference" section to go see that AWS backend and see the endpoint we created

Endpoint: sagemaker-scikit-learn-2026-02-22-18-39-09-933
Model data: s3://sagemaker-us-east-2-047557961235/sagemaker-scikit-learn-2026-02-22-18-14-53-105/output/model.tar.gz


In [44]:
# Sample values for the 11 input columns, important they are in same order
# Let's use the median values for each feature again here and confirm the output is the same as on the Heroku website interface we built as part of problem 1 I suppose...
sample = [[7.9, 0.52, 0.26, 2.2, 0.079, 14, 38, 0.99675, 3.31, 0.62, 10.2]]
pred = predictor_2a.predict(sample)
print("Prediction:", pred)

# Yep! Looks good, matches (though of course more precise here as we round in the html output)

Prediction: [5.5679515]


In [45]:
# Apparently this can be run after done running everything to stop the charges
# Still need to understand how $ charges even work in this platform... 
# I never gave a cc or anything yet either? Where do we even go view charges? 
# Investigate further later once this initial training .ipynb notebook is working
# Oh! The video from 2/19 I think will be a tremendous resource now as well 
# Reviewed the other 2 linked in the content already, but his actual lecture video is found here
# https://navigate.nu.edu/d2l/lms/news/main.d2l?ou=59967

# Okay better understanding of eveyrthing now though still need to keep an eye on charges. Assignment dictates to have a live site on Heroku so I think we can't clean yet
# Once the assignment is graded, I'll need to remember topop back in here though I think.... and run this cell to kill this endpoint to stop the ongoing charges
# Oh wait he does indicate in the video not to forget to shut down your notebook and delete your endpoints! And so perhaps we don't need to keep this alive then?
# Ahhh yes upon revisiting the instructions I see now that we only need the live URL on Herku for problem 1... for problem 2 he only wants the notebook submited on GitHub
# i.e. I think for this second problem where we're using AWS/SageMaker for a and b (without and with docker, respectively) we do NOT need a live site so I'll kill it now
predictor_2a.delete_endpoint()
print("Endpoint deleted")


INFO:sagemaker:Deleting endpoint configuration with name: sagemaker-scikit-learn-2026-02-22-18-39-09-933
INFO:sagemaker:Deleting endpoint with name: sagemaker-scikit-learn-2026-02-22-18-39-09-933


Endpoint deleted


In [46]:
# Some code which should allow us to check for all endpoints (showing 1 now, rerun once the endpoint is deleted and then showing 0)
import boto3
sm = boto3.client("sagemaker")
for ep in sm.list_endpoints(SortBy="CreationTime", SortOrder="Descending")["Endpoints"][:20]:
    print(ep["EndpointName"], ep["EndpointStatus"])


In [ ]:
# Okay cool! No output from the above now post-deletion of the endpoint. So part a of problem 2 of assignment 5 in week 3 is done
# Now moving on to part B where we'll basically do the same thing but now using container technology i.e. Docker (instead of part a did not use Docker)

In [ ]:
# Part B setup for custom container flow (BYOC)
import boto3
import sagemaker
from sagemaker.estimator import Estimator
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

session = sagemaker.Session()
region = session.boto_region_name
bucket = session.default_bucket()
role = "arn:aws:iam::047557961235:role/service-role/AmazonSageMaker-ExecutionRole-20260221T114423"
account_id = boto3.client("sts").get_caller_identity()["Account"]

repo_name = "wine-quality-2b"
image_tag = "latest"
image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{repo_name}:{image_tag}"

print("region:", region)
print("bucket:", bucket)
print("account_id:", account_id)
print("image_uri:", image_uri)


In [ ]:
# Create ECR repo if it does not already exist
ecr = boto3.client("ecr", region_name=region)
try:
    ecr.describe_repositories(repositoryNames=[repo_name])
    print("ECR repo already exists")
except ecr.exceptions.RepositoryNotFoundException:
    ecr.create_repository(repositoryName=repo_name)
    print("ECR repo created")


In [ ]:
# Build and push the container from this notebook instance
!chmod +x container/wine_quality/train container/wine_quality/serve
!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com
!docker build -t {repo_name}:{image_tag} ./container/wine_quality
!docker tag {repo_name}:{image_tag} {image_uri}
!docker push {image_uri}


In [ ]:
# Define estimator for Part B using our custom image (container tech)
custom_estimator = Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m4.xlarge",
    output_path=f"s3://{bucket}/ana680/wine-quality/models-2b",
    sagemaker_session=session,
)

print(custom_estimator)


In [ ]:
# Train with the same S3 training data used in Part A
custom_estimator.fit({"train": train_s3_uri})
print("Part B training complete")


In [ ]:
# Save references so we can compare/cleanup by explicit names
print("2B training job:", custom_estimator.latest_training_job.name)
print("2B model data:", custom_estimator.model_data)


In [ ]:
# Deploy endpoint for Part B custom container
# Using t2.medium here since we already used that successfully for endpoint usage in Part A
predictor_2b = custom_estimator.deploy(initial_instance_count=1, instance_type="ml.t2.medium")
predictor_2b.serializer = CSVSerializer()
predictor_2b.deserializer = JSONDeserializer()
print("2B endpoint:", predictor_2b.endpoint_name)


In [ ]:
# Test with same sample row as before so we can confirm easily again that we get the same output
sample = [[7.9, 0.52, 0.26, 2.2, 0.079, 14, 38, 0.99675, 3.31, 0.62, 10.2]]
pred_2b = predictor_2b.predict(sample)
print("2B prediction:", pred_2b)


In [ ]:
# Cleanup endpoint right after test to stop charges again
predictor_2b.delete_endpoint()
print("2B endpoint deleted")


In [ ]:
# Final good gut-check again to ensure list of all endpoints has none; to prevent accidental cost
for ep in sm.list_endpoints(SortBy="CreationTime", SortOrder="Descending")["Endpoints"][:20]:
    print(ep["EndpointName"], ep["EndpointStatus"])
